# Experiment H16: LR Confound Audit for Final Spectral Metrics

**Counterpart script:** `experiments/Tier_4_Falsification_Stress_Tests/H16_LR_CONFOUND_SPECTRAL_DEMOCRACY/run_experiment.py`

This notebook is the presentation companion to the H16 script and intentionally **imports the shared script implementation** rather than maintaining a separate numerical fork.

## Scope of this pair

This H16 pair audits learning-rate confounds for **final effective rank** and **final condition number** only.

- **Computed here:** final loss, final effective rank per layer, final condition number per layer.
- **Not computed here:** WeightWatcher alpha, power-law tail fits, or any other alpha-style spectral-tail metric.

That alpha wording previously present in the H16 script was stale. In this first serious completion pass, the pair is explicitly narrowed to the metrics it actually computes.


## Research question and protocol

**Question.** If Muon and SGD are compared at matched learning rates chosen from the same coarse grid-search protocol, do the final-state spectral conclusions change materially?

**Protocol.**
1. Sweep candidate learning rates separately for SGD and Muon on the first 3 seeds.
2. Choose the grid-best LR by lowest mean final loss on that 3-seed subset.
3. Evaluate four configurations across 10 seeds:
   - SGD @ default LR
   - SGD @ chosen LR
   - Muon @ default LR
   - Muon @ chosen LR
4. Record the final loss, final effective rank, and final condition number for every converged seed.
5. Report the original heuristic falsification checks:
   - **T1:** Does the mean layerwise Muon-SGD effective-rank difference change by more than 50%?
   - **T2:** Does the mean layerwise Muon/SGD condition-number ratio flip across 1?

This is a **final-state LR-confound audit**, not a full dynamical or alpha-analysis study.


In [ ]:
from pathlib import Path
import importlib
import sys
import time

import numpy as np
import matplotlib.pyplot as plt


def find_experiment_dir():
    target_rel = Path('experiments/Tier_4_Falsification_Stress_Tests/H16_LR_CONFOUND_SPECTRAL_DEMOCRACY')
    search_roots = [Path.cwd(), *Path.cwd().parents]
    for base in search_roots:
        direct = base / 'run_experiment.py'
        if direct.exists() and base.name == target_rel.name:
            return base.resolve()
        repo_relative = base / target_rel / 'run_experiment.py'
        if repo_relative.exists():
            return repo_relative.parent.resolve()
    raise FileNotFoundError('Could not locate H16 experiment directory from the current working directory.')


EXPERIMENT_DIR = find_experiment_dir()
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import run_experiment as h16
h16 = importlib.reload(h16)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

EXPERIMENT_DIR


## Reproducibility and configuration

The notebook uses the exact configuration exported by the script. The printout below makes the scope and runtime-relevant settings explicit, including the fact that "optimal LR" means **best LR on a fixed candidate grid using only the first 3 seeds**.


In [ ]:
config_preview = {
    'experiment_title': h16.EXPERIMENT_TITLE,
    'script_path': str(EXPERIMENT_DIR / 'run_experiment.py'),
    'notebook_path': str(EXPERIMENT_DIR / 'run_experiment.ipynb'),
    'scope_note': h16.SCOPE_NOTE,
    'dim': h16.DIM,
    'num_layers': h16.NUM_LAYERS,
    'num_steps': h16.NUM_STEPS,
    'batch_size': h16.BATCH_SIZE,
    'momentum': h16.MOMENTUM,
    'ns_iters': h16.NS_ITERS,
    'num_seeds': h16.NUM_SEEDS,
    'lr_sweep_num_seeds': h16.LR_SWEEP_NUM_SEEDS,
    'default_lrs': {'sgd': h16.DEFAULT_SGD_LR, 'muon': h16.DEFAULT_MUON_LR},
    'candidate_lrs': {'sgd': h16.SGD_LRS, 'muon': h16.MUON_LRS},
    'seeds': h16.get_default_seeds(),
    'weight_seed_offset': h16.WEIGHT_SEED_OFFSET,
}

for key, value in config_preview.items():
    print(f'{key}: {value}')


## Execute the shared experiment implementation

This cell runs the shared script logic and captures the structured results dictionary used throughout the rest of the notebook.


In [ ]:
start_time = time.perf_counter()
results = h16.run_experiment()
runtime_seconds = time.perf_counter() - start_time

print(results['title'])
print(results['scope_note'])
print(f"Runtime: {runtime_seconds:.2f} seconds")
print(f"Chosen LR (SGD):  {results['lr_sweep']['sgd']['best_lr']:.4f}")
print(f"Chosen LR (Muon): {results['lr_sweep']['muon']['best_lr']:.4f}")
assert results['scope']['alpha_computed'] is False


## Helper utilities for tables and figures

These helpers are presentation-only. The numerical experiment itself lives in the imported script.


In [ ]:
CONFIG_ORDER = results['config_order']
LABELS = {name: results['configs'][name]['label'] for name in CONFIG_ORDER}
COLORS = {
    'sgd_default': '#4C72B0',
    'sgd_optimal': '#55A868',
    'muon_default': '#C44E52',
    'muon_optimal': '#8172B2',
}
JITTER_RNG = np.random.RandomState(0)


def format_number(value, precision=4):
    if value is None:
        return 'NA'
    value = float(value)
    if not np.isfinite(value):
        return 'inf' if value > 0 else '-inf'
    if value == 0:
        return '0'
    if abs(value) >= 1e3 or abs(value) < 1e-3:
        return f'{value:.3e}'
    return f'{value:.{precision}f}'


def print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    for row in rows:
        widths = [max(widths[i], len(str(row[i]))) for i in range(len(headers))]
    print(' | '.join(str(headers[i]).ljust(widths[i]) for i in range(len(headers))))
    print('-+-'.join('-' * widths[i] for i in range(len(headers))))
    for row in rows:
        print(' | '.join(str(row[i]).ljust(widths[i]) for i in range(len(headers))))


def metric_matrix(config_result, metric_name):
    key = f'{metric_name}_by_layer'
    rows = [seed_result[key] for seed_result in config_result['seed_results'] if seed_result[key] is not None]
    if not rows:
        return np.zeros((0, h16.NUM_LAYERS), dtype=float)
    return np.asarray(rows, dtype=float)


def print_lr_sweep_table(results):
    for optimizer in ['sgd', 'muon']:
        sweep = results['lr_sweep'][optimizer]
        rows = []
        for entry in sweep['results']:
            rows.append([
                optimizer.upper(),
                f"{entry['lr']:.4f}",
                format_number(entry['mean_final_loss']),
                entry['finite_runs'],
                entry['diverged_runs'],
            ])
        print(f'\n{optimizer.upper()} LR sweep (selection subset: {sweep["seed_subset"]})')
        print_table(['Optimizer', 'LR', 'Mean final loss', 'Finite', 'Diverged'], rows)
        print(f"Chosen LR: {sweep['best_lr']:.4f}\n")


def plot_lr_sweep(results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, optimizer in zip(axes, ['sgd', 'muon']):
        sweep = results['lr_sweep'][optimizer]
        entries = sorted(sweep['results'], key=lambda item: item['lr'])
        lrs = np.array([entry['lr'] for entry in entries], dtype=float)
        means = np.array([entry['mean_final_loss'] for entry in entries], dtype=float)
        default_lr = results['config']['default_lrs'][optimizer]
        best_lr = sweep['best_lr']
        default_entry = next(entry for entry in entries if abs(entry['lr'] - default_lr) < 1e-12)
        best_entry = next(entry for entry in entries if abs(entry['lr'] - best_lr) < 1e-12)

        ax.plot(lrs, means, marker='o', color='#333333', linewidth=2)
        ax.scatter(default_entry['lr'], default_entry['mean_final_loss'], s=100, marker='s', color='#C44E52', label='default LR', zorder=3)
        ax.scatter(best_entry['lr'], best_entry['mean_final_loss'], s=180, marker='*', color='#55A868', label='chosen LR', zorder=4)
        ax.set_xscale('log')
        ax.set_xlabel('Learning rate')
        ax.set_ylabel('Mean final loss on 3-seed sweep')
        ax.set_title(f'{optimizer.upper()} LR sweep')
        ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def print_loss_summary_table(results):
    rows = []
    for name in CONFIG_ORDER:
        config_result = results['configs'][name]
        summary = config_result['summary']
        rows.append([
            LABELS[name],
            config_result['optimizer'].upper(),
            f"{config_result['lr']:.4f}",
            format_number(summary['loss_mean_finite']),
            format_number(summary['loss_sd_finite']),
            format_number(summary['loss_sem_finite']),
            summary['finite_runs'],
            summary['diverged_runs'],
        ])
    print_table(
        ['Config', 'Optimizer', 'LR', 'Mean final loss', 'SD', 'SEM', 'Finite', 'Diverged'],
        rows,
    )


def plot_seed_losses(results):
    fig, ax = plt.subplots(figsize=(11, 5))
    positions = np.arange(1, len(CONFIG_ORDER) + 1)
    data = []
    for name in CONFIG_ORDER:
        losses = [seed_result['final_loss'] for seed_result in results['configs'][name]['seed_results'] if np.isfinite(seed_result['final_loss'])]
        data.append(losses)

    box = ax.boxplot(data, positions=positions, widths=0.55, patch_artist=True, showfliers=False)
    for patch, name in zip(box['boxes'], CONFIG_ORDER):
        patch.set_facecolor(COLORS[name])
        patch.set_alpha(0.25)

    for pos, name, losses in zip(positions, CONFIG_ORDER, data):
        jitter = JITTER_RNG.normal(loc=0.0, scale=0.04, size=len(losses))
        ax.scatter(np.full(len(losses), pos) + jitter, losses, color=COLORS[name], s=35, alpha=0.75)
        diverged = results['configs'][name]['summary']['diverged_runs']
        ymax = max(losses) if losses else 0.0
        ax.text(pos, ymax + 0.01, f'diverged: {diverged}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(positions)
    ax.set_xticklabels([LABELS[name].replace(' @ ', '\n@ ') for name in CONFIG_ORDER])
    ax.set_ylabel('Final loss')
    ax.set_title('Seed-level final loss across default vs chosen LR configurations')
    plt.tight_layout()
    plt.show()


def print_metric_summary_table(results, metric_name, title):
    rows = []
    for name in CONFIG_ORDER:
        summary = results['configs'][name]['summary']
        means = summary[f'{metric_name}_mean_by_layer']
        sems = summary[f'{metric_name}_sem_by_layer']
        row = [LABELS[name]]
        for layer_idx in range(h16.NUM_LAYERS):
            row.append(f"{format_number(means[layer_idx], 2)} ± {format_number(sems[layer_idx], 2)}")
        row.append(f"{summary['finite_runs']}/{summary['num_seeds']}")
        rows.append(row)
    print(title)
    print_table(['Config'] + [f'L{layer_idx} mean ± SEM' for layer_idx in range(h16.NUM_LAYERS)] + ['Finite'], rows)


def plot_layerwise_metric(results, metric_name, ylabel, title, yscale='linear'):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    x = np.arange(h16.NUM_LAYERS, dtype=float)
    offsets = np.linspace(-0.24, 0.24, len(CONFIG_ORDER))

    for offset, name in zip(offsets, CONFIG_ORDER):
        config_result = results['configs'][name]
        summary = config_result['summary']
        matrix = metric_matrix(config_result, metric_name)
        means = np.asarray(summary[f'{metric_name}_mean_by_layer'], dtype=float)
        sems = np.asarray([0.0 if value is None else value for value in summary[f'{metric_name}_sem_by_layer']], dtype=float)

        for row in matrix:
            jitter = JITTER_RNG.normal(loc=0.0, scale=0.012, size=row.shape[0])
            ax.scatter(x + offset + jitter, row, color=COLORS[name], alpha=0.18, s=20)

        ax.errorbar(
            x + offset,
            means,
            yerr=sems,
            marker='o',
            capsize=4,
            linewidth=2,
            color=COLORS[name],
            label=LABELS[name],
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f'L{layer_idx}' for layer_idx in range(h16.NUM_LAYERS)])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_yscale(yscale)
    ax.legend(frameon=False, ncol=2)
    plt.tight_layout()
    plt.show()


def print_test_summary(results):
    t1 = results['tests']['t1']
    t2 = results['tests']['t2']
    rows = [
        [
            'T1',
            'Effective-rank difference changes by >50%',
            format_number(t1['default_difference_mean']),
            format_number(t1['optimal_difference_mean']),
            format_number(t1['threshold']),
            'PASS' if t1['passed'] else 'FAIL',
        ],
        [
            'T2',
            'Muon/SGD condition-number ratio flips across 1',
            format_number(t2['default_muon_over_sgd_ratio_mean']),
            format_number(t2['optimal_muon_over_sgd_ratio_mean']),
            '1.0 boundary',
            'PASS' if t2['passed'] else 'FAIL',
        ],
    ]
    print_table(['Test', 'Question', 'Default', 'Chosen-LR', 'Threshold', 'Verdict'], rows)
    print('')
    print('T1 interpretation:', t1['interpretation'])
    print('T2 interpretation:', t2['interpretation'])


## Learning-rate sweep comparison

The figures and tables below show the coarse LR search actually used to define the notebook's "chosen LR" language. This is deliberately presented as a **grid-best LR on 3 seeds**, not as a globally optimized or statistically exhaustive hyperparameter search.


In [ ]:
print_lr_sweep_table(results)
plot_lr_sweep(results)


### Interpretation: LR sweep

- The chosen LR is the candidate with the lowest **mean final loss** on the 3-seed sweep subset.
- Any downstream comparison in this notebook therefore reflects **matched coarse-grid training quality**, not a claim about exact optimizer-optimal tuning.
- Because the sweep uses only 3 seeds and a fixed candidate list, small changes in the candidate grid could change the selected LR, especially when losses are close.


## Seed-level final loss comparison

Final loss is shown first because the spectral metrics are being interpreted under an LR-matching argument. If the chosen-LR and default-LR configurations achieve nearly identical losses, then large spectral shifts are especially suggestive of an LR confound rather than a gross optimization-quality gap.


In [ ]:
print_loss_summary_table(results)
plot_seed_losses(results)


### Interpretation: final loss

This panel should be read together with the LR sweep figure:

- It shows whether the chosen-LR comparison materially changes final training quality across seeds.
- Divergence counts are reported explicitly rather than being silently hidden.
- In this experiment, loss differences are relatively modest, so any large movement in spectral summaries deserves attention rather than being dismissed as a trivial consequence of catastrophic training failure.


## Layerwise effective rank (spectral democracy)

Each point is one converged seed, while the line markers show the mean ± SEM across seeds for each layer. This is the main visualization for the "spectral democracy" part of H16.


In [ ]:
print_metric_summary_table(results, 'effective_rank', 'Effective rank summary (mean ± SEM)')
plot_layerwise_metric(
    results,
    metric_name='effective_rank',
    ylabel='Effective rank',
    title='Layerwise final effective rank across default vs chosen LR configurations',
    yscale='linear',
)


### Interpretation: effective rank

The relevant question is not simply whether Muon is above or below SGD at one default LR, but whether the **Muon-SGD gap remains stable after LR matching**. The T1 heuristic below formalizes that question using the same >50% change criterion as the script.


## Layerwise condition number

Condition number spans a much wider numerical range than effective rank, so the figure uses a **log y-scale**. As above, points show individual converged seeds and line markers show the mean ± SEM by layer.


In [ ]:
print_metric_summary_table(results, 'condition_number', 'Condition number summary (mean ± SEM)')
plot_layerwise_metric(
    results,
    metric_name='condition_number',
    ylabel='Condition number',
    title='Layerwise final condition number across default vs chosen LR configurations',
    yscale='log',
)


### Interpretation: condition number

The condition-number comparison is especially sensitive to the LR-confound framing because a qualitative ranking flip can occur even when final losses are fairly close. The T2 heuristic below captures exactly that question by testing whether the mean Muon/SGD condition-number ratio crosses 1 after LR matching.


## Final T1/T2 verdicts

The table below reports the same pair-level falsification heuristics used by the script, now with the intermediate quantities made explicit.


In [ ]:
print_test_summary(results)


## Conclusion and limitations

### Pair-level conclusion

This first completion pass makes the H16 pair internally consistent:

- the notebook and script now share one implementation,
- the pair explicitly states that **alpha is not computed here**, and
- the LR-confound story is presented with seed-level losses, layerwise spectral summaries, and explicit T1/T2 verdicts.

### Important limitations

1. **Coarse LR grid.** "Chosen LR" means the best candidate on a fixed hand-written grid, not a continuously optimized LR.
2. **Only 3 seeds for LR selection.** The sweep is intentionally lightweight and therefore noisy.
3. **Final-state metrics only.** This notebook does not analyze trajectories over training time.
4. **Condition-number averaging can be outlier-sensitive.** The arithmetic mean of kappa is useful for continuity with the original heuristic but should not be overinterpreted.
5. **Alpha remains out of scope.** Any WeightWatcher-style or power-law-tail claim requires a separate implementation and verification pass.

### Recommended next verification

- Re-run both the script and notebook after any future edit to ensure the notebook still matches the script.
- If stronger inferential claims are needed, add bootstrap confidence intervals or permutation-style checks on the seed-level summaries.
- If alpha is later required, treat it as a distinct extension rather than smuggling it into the present LR-confound audit.
